# Adversarial MRI rerun pipeline (Colab)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ota0425/brain-tumor-adversarial-experiment/blob/main/rerun_colab.ipynb)

This notebook runs the reproducible pipeline supplied in `MyDrive/ThammasatResearch/rerun`. Existing experiments must be stored in `results_before_rerun`; all outputs from this run are written to `results`.

Select **Runtime > Change runtime type > T4 GPU**, then run the cells in order. Completed stages are skipped, so **Run all** does not retrain existing models. Do not use `--force` unless intentionally starting the entire pipeline again.

In [1]:
%pip install -q "tensorflow==2.20.*" scikit-learn pandas matplotlib pillow

## 1. Mount Drive and validate the workspace

In [2]:
from google.colab import drive
drive.mount("/content/drive")

import hashlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

BASE = Path("/content/drive/MyDrive/ThammasatResearch")
RERUN = BASE / "rerun"
DATASET = BASE / "dataset" / "archive.zip"
MODELS = BASE / "models"
RESULTS = BASE / "results"
RESULTS_BEFORE = BASE / "results_before_rerun"

os.environ["TR_BASE"] = str(BASE)
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
MODELS.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

required = [DATASET, RERUN / "common.py", RERUN / "01_train_classifier.py", RERUN / "06b_intersection_analysis.py"]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))

print("TR_BASE:", BASE)
print("Dataset:", DATASET)
print("Output directory:", RESULTS)

Mounted at /content/drive
TR_BASE: /content/drive/MyDrive/ThammasatResearch
Dataset: /content/drive/MyDrive/ThammasatResearch/dataset/archive.zip
Output directory: /content/drive/MyDrive/ThammasatResearch/results


## 2. Recover the completed Stage 1 record if necessary

If Stage 1 was already run before `results` was renamed, its manifest and reports are copied from `results_before_rerun` into the new `results`. The model hash is verified before anything continues.

In [3]:
classifier = MODELS / "classifier_seed42.keras"
manifest_path = RESULTS / "manifest.json"
stage1_files = ["manifest.json", "classifier_history.json", "classifier_test_report.json"]

if classifier.exists() and not manifest_path.exists():
    old_manifest = RESULTS_BEFORE / "manifest.json"
    if not old_manifest.exists():
        raise RuntimeError(
            "classifier_seed42.keras exists, but no manifest was found in results or "
            "results_before_rerun. Do not retrain or continue until this is resolved."
        )
    for name in stage1_files:
        source = RESULTS_BEFORE / name
        destination = RESULTS / name
        if source.exists() and not destination.exists():
            shutil.copy2(source, destination)
            print("Recovered:", destination)

if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if classifier.exists() and "classifier_sha256" in manifest:
        digest = hashlib.sha256(classifier.read_bytes()).hexdigest()
        if digest != manifest["classifier_sha256"]:
            raise RuntimeError("Classifier SHA-256 does not match results/manifest.json.")
        print("Verified existing classifier:", digest)
        print("Recorded clean accuracy:", manifest.get("clean_test_accuracy"))
else:
    print("No completed Stage 1 was found; Stage 1 will run below.")

Verified existing classifier: 90638f73f6ec76c195c8e2e4c787580c1b9c34d21867f5b3f9710778fc424728
Recorded clean accuracy: 0.819375


## 3. Environment record and safe stage runner

In [4]:
import tensorflow as tf

print("Python:", sys.version)
print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
if not tf.__version__.startswith("2.20."):
    raise RuntimeError("TensorFlow 2.20.x is required. Restart the runtime and run again.")
if not tf.config.list_physical_devices("GPU"):
    raise RuntimeError("No GPU detected. Select a T4 GPU runtime before continuing.")

def run_stage(script, outputs):
    outputs = [BASE / output for output in outputs]
    if all(path.exists() for path in outputs):
        print(f"SKIPPED {script}: output files already exist.")
        for path in outputs:
            print(" -", path)
        return
    completed = subprocess.run(
        [sys.executable, script], cwd=RERUN, env=os.environ.copy(), check=False
    )
    if completed.returncode != 0:
        raise RuntimeError(f"{script} failed with exit code {completed.returncode}.")
    absent = [str(path) for path in outputs if not path.exists()]
    if absent:
        raise RuntimeError("Stage finished without expected outputs: " + ", ".join(absent))
    print(f"COMPLETED {script}")

Python: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
TensorFlow: 2.20.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 4. Main pipeline

Run these cells in order. Training stages are skipped when their verified artifacts already exist.

In [5]:
# Stage 1 — classifier training
run_stage("01_train_classifier.py", [
    "models/classifier_seed42.keras",
    "results/manifest.json",
    "results/classifier_test_report.json",
])

SKIPPED 01_train_classifier.py: output files already exist.
 - /content/drive/MyDrive/ThammasatResearch/models/classifier_seed42.keras
 - /content/drive/MyDrive/ThammasatResearch/results/manifest.json
 - /content/drive/MyDrive/ThammasatResearch/results/classifier_test_report.json


In [6]:
# Stage 2 — FGSM sweep
run_stage("02_fgsm_sweep.py", ["results/fgsm_summary.csv"])

SKIPPED 02_fgsm_sweep.py: output files already exist.
 - /content/drive/MyDrive/ThammasatResearch/results/fgsm_summary.csv


In [7]:
# Stages 3–4 — detector v2 and final test
run_stage("03_detector_v2.py", [
    "models/detector_v2_seed42.keras",
    "results/detector_v2_threshold.json",
])
run_stage("04_final_test_v2.py", ["results/detector_v2_test_by_epsilon.csv"])

SKIPPED 03_detector_v2.py: output files already exist.
 - /content/drive/MyDrive/ThammasatResearch/models/detector_v2_seed42.keras
 - /content/drive/MyDrive/ThammasatResearch/results/detector_v2_threshold.json
SKIPPED 04_final_test_v2.py: output files already exist.
 - /content/drive/MyDrive/ThammasatResearch/results/detector_v2_test_by_epsilon.csv


In [8]:
# Stages 5–6 — calibrated detector v2b and final test
run_stage("03b_detector_v2_calibrated.py", [
    "models/detector_v2b_seed42.keras",
    "results/detector_v2b_threshold.json",
])
run_stage("04b_final_test_v2b.py", ["results/detector_v2b_test_by_epsilon.csv"])

SKIPPED 03b_detector_v2_calibrated.py: output files already exist.
 - /content/drive/MyDrive/ThammasatResearch/models/detector_v2b_seed42.keras
 - /content/drive/MyDrive/ThammasatResearch/results/detector_v2b_threshold.json
SKIPPED 04b_final_test_v2b.py: output files already exist.
 - /content/drive/MyDrive/ThammasatResearch/results/detector_v2b_test_by_epsilon.csv


In [9]:
# Stage 7 — deployment calibration
run_stage("05_deployment_calibration.py", [
    "results/deployment_calibration_threshold.json",
    "results/detector_v2b_deploycal_eval_by_epsilon.csv",
])

SKIPPED 05_deployment_calibration.py: output files already exist.
 - /content/drive/MyDrive/ThammasatResearch/results/deployment_calibration_threshold.json
 - /content/drive/MyDrive/ThammasatResearch/results/detector_v2b_deploycal_eval_by_epsilon.csv


In [10]:
# Stage 8 — PGD evaluation (about 20 minutes on an RTX 3070)
run_stage("06_pgd_eval.py", ["results/pgd_eval_by_epsilon.csv"])

SKIPPED 06_pgd_eval.py: output files already exist.
 - /content/drive/MyDrive/ThammasatResearch/results/pgd_eval_by_epsilon.csv


In [11]:
# Stage 9 — FGSM/PGD intersection analysis (about 20 minutes on an RTX 3070)
run_stage("06b_intersection_analysis.py", [
    "results/attack_intersection_analysis.csv",
    "results/attack_scores.npz",
])

SKIPPED 06b_intersection_analysis.py: output files already exist.
 - /content/drive/MyDrive/ThammasatResearch/results/attack_intersection_analysis.csv
 - /content/drive/MyDrive/ThammasatResearch/results/attack_scores.npz


## 5. Review generated results

In [12]:
print("Generated result files:")
for path in sorted(RESULTS.iterdir()):
    print(f"{path.name:55s} {path.stat().st_size:>12,d} bytes")

manifest = json.loads((RESULTS / "manifest.json").read_text(encoding="utf-8"))
print("\nManifest:")
print(json.dumps(manifest, indent=2))

Generated result files:
attack_intersection_analysis.csv                               1,823 bytes
attack_scores.npz                                             98,993 bytes
classifier_history.json                                          942 bytes
classifier_test_report.json                                    1,274 bytes
deployment_calibration_threshold.json                            472 bytes
detector_v2_history.csv                                        2,171 bytes
detector_v2_test_by_epsilon.csv                                1,085 bytes
detector_v2_threshold.json                                       374 bytes
detector_v2b_deploycal_eval_by_epsilon.csv                       979 bytes
detector_v2b_history.csv                                       2,305 bytes
detector_v2b_test_by_epsilon.csv                               1,241 bytes
detector_v2b_threshold.json                                      658 bytes
fgsm_summary.csv                                                 734 bytes
m